In [ ]:
# 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 기본 라이브러리
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## csv 파일 합치기

아래 참조

| video_id | frame_idx | timestamp | kp0_x | kp0_y | kp0_conf | kp1_x | kp1_y | kp1_conf | kp2_x | kp2_y | kp2_conf | … |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| walk01 | 0 | 0.000 | 0.55 | 0.30 | 0.99 | 0.60 | 0.50 | 0.95 | 0.45 | 0.80 | 0.93 | … |
| walk01 | 1 | 0.033 | 0.56 | 0.31 | 0.98 | 0.61 | 0.51 | 0.95 | 0.46 | 0.80 | 0.94 | … |
| walk01 | 2 | 0.066 | 0.58 | 0.32 | 0.97 | 0.63 | 0.52 | 0.96 | 0.47 | 0.81 | 0.95 | … |

In [ ]:
# landmark_index 기준으로 DF 다시 설정하는 함수
def pivot_landmarks(df: pd.DataFrame) -> pd.DataFrame:
    df_wide = df.pivot(
        index=['video', 'file_id', 'frame', 'timestamp'],
        columns='landmark_index',
        values=['x', 'y', 'z', 'visibility']
    )

    df_wide.columns = [f'kp{col[1]}_{col[0]}' for col in df_wide.columns]
    df_wide.reset_index(inplace=True)
    cols = ['video', 'file_id', 'frame', 'timestamp']
    kp_cols = sorted(
        [col for col in df_wide.columns if col not in cols],
        key=lambda x: (int(x.split('_')[0][2:]), x.split('_')[1])
    )
    return df_wide[cols + kp_cols]

In [ ]:
# 모든 파일 불러오기
csv_dir = "/content/drive/MyDrive/OnSafe/fms_clean_dataset/pose_csv_results"
csv_paths = glob.glob(os.path.join(csv_dir, '**', '*.csv'), recursive=True)


In [ ]:
# 결과 모을 빈 리스트 할당
all_dfs = []

# 각 비디오별 처리 반복
for csv_path in csv_paths:
    print(f"Processing: {csv_path}")

    df = pd.read_csv(csv_path) # 파일 df 형식으로 읽기
    df_wide = pivot_landmarks(df) # 함수 사용
    all_dfs.append(df_wide) # 리스트에 추가하기


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
Processing: /content/drive/MyDrive/OnSafe/fms_clean_dataset/pose_csv_results/FALL/C_N_180_resized_clip_00_pose.csv
Processing: /content/drive/MyDrive/OnSafe/fms_clean_dataset/pose_csv_results/FALL/C_N_180_resized_clip_01_pose.csv
Processing: /content/drive/MyDrive/OnSafe/fms_clean_dataset/pose_csv_results/FALL/C_D_0083_clip_00_pose.csv
Processing: /content/drive/MyDrive/OnSafe/fms_clean_dataset/pose_csv_results/FALL/C_D_0083_clip_01_pose.csv
Processing: /content/drive/MyDrive/OnSafe/fms_clean_dataset/pose_csv_results/FALL/C_D_0037_clip_00_pose.csv
Processing: /content/drive/MyDrive/OnSafe/fms_clean_dataset/pose_csv_results/FALL/C_D_0037_clip_01_pose.csv
Processing: /content/drive/MyDrive/OnSafe/fms_clean_dataset/pose_csv_results/FALL/C_D_0037_clip_02_pose.csv
Processing: /content/drive/MyDrive/OnSafe/fms_clean_dataset/pose_csv_results/FALL/C_D_0037_clip_03_pose.csv
Processing: /content/drive/MyDrive/OnSafe/fms_clean_dataset/pose_csv_results/FALL/C_D_

In [ ]:
# 하나의 csv 파일로 변환
result_df = pd.concat(all_dfs, ignore_index=True)
result_df


result_df.to_csv("/content/drive/MyDrive/OnSafe/all_clean_df_concat.csv", index=False, encoding='utf-8')

## 노이즈 제거 및 보전 (결측치/이상치 처리)



*   신뢰도가 너무 낮거나 좌표가 이상치일 경우 -> 이전&다음 프레임 좌표로 보강
*   결측치 보강 : kp[i] = (kp[i-1]+kp[i+1])/2



In [ ]:
def denoise_kp_dataframe_3d(df, conf_threshold=0.3):
    """
    3D 키포인트(x,y,z)에서 신뢰도(visibility) 낮거나 이상치인 좌표를 이전/다음 프레임 평균으로 보강

    Args:
      df : DataFrame
        ['video_id','frame_idx','timestamp', 'kp0_x','kp0_y','kp0_z','kp0_visibility', ...]
      conf_threshold : float
        - 신뢰도 임계값(default=0.3)

    Returns:
      df_processed : DataFrame
        - 노이즈 제거 및 보강 완료
    """
    # DataFrame 복사
    df_processed = df.copy()

    # 존재하는 키포인트 수
    kp_x_cols = sorted([c for c in df.columns if '_x' in c])
    kp_y_cols = sorted([c for c in df.columns if '_y' in c])
    kp_z_cols = sorted([c for c in df.columns if '_z' in c])
    conf_cols = sorted([c for c in df.columns if '_visibility' in c])

    num_joints = len(kp_x_cols)
    num_frames = df.shape[0]

    # numpy 배열로 변환
    kp = np.zeros((num_frames, num_joints, 3))
    conf = np.zeros((num_frames, num_joints))

    for j in range(num_joints):
        kp[:, j, 0] = df[kp_x_cols[j]]
        kp[:, j, 1] = df[kp_y_cols[j]]
        kp[:, j, 2] = df[kp_z_cols[j]]
        conf[:, j] = df[conf_cols[j]]

    # 신뢰도 낮은 좌표 → NaN
    kp[conf < conf_threshold] = np.nan

    # 이상치 탐지 (평균 ± 3*std)
    mean = np.nanmean(kp, axis=(0,1))
    std = np.nanstd(kp, axis=(0,1))
    outlier_mask = (kp < mean - 3*std) | (kp > mean + 3*std)
    kp[outlier_mask] = np.nan

    # NaN 보강 (이전·다음 프레임 평균 or 단일 값)
    for f in range(num_frames):
        for j in range(num_joints):
            if np.isnan(kp[f, j, 0]):
                prev_val, next_val = None, None

                # 이전 값 찾기
                prev = f - 1
                while prev >= 0:
                    if not np.isnan(kp[prev, j, 0]):
                        prev_val = kp[prev, j, :]
                        break
                    prev -= 1

                # 다음 값 찾기
                next_f = f + 1
                while next_f < num_frames:
                    if not np.isnan(kp[next_f, j, 0]):
                        next_val = kp[next_f, j, :]
                        break
                    next_f += 1

                # 보강
                if prev_val is not None and next_val is not None:
                    kp[f, j, :] = (prev_val + next_val) / 2
                elif prev_val is not None:
                    kp[f, j, :] = prev_val
                elif next_val is not None:
                    kp[f, j, :] = next_val
                # 양쪽 모두 NaN이면 그대로 NaN 유지

    # numpy 배열 → DataFrame
    for j in range(num_joints):
        df_processed[kp_x_cols[j]] = kp[:, j, 0]
        df_processed[kp_y_cols[j]] = kp[:, j, 1]
        df_processed[kp_z_cols[j]] = kp[:, j, 2]
        # confidence 컬럼은 그대로 유지

    return df_processed


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/OnSafe/all_clean_df_concat.csv")
df_processed = denoise_kp_dataframe_3d(df, conf_threshold=0.3)

df_processed


,video,file_id,frame,timestamp,kp0_visibility,kp0_x,kp0_y,kp0_z,kp1_visibility,kp1_x,...,kp30_y,kp30_z,kp31_visibility,kp31_x,kp31_y,kp31_z,kp32_visibility,kp32_x,kp32_y,kp32_z
0,ADL,ADL_258_clip_06,0,0.000000,0.998187,0.796604,0.774230,-0.013695,0.998785,0.792341,...,0.520401,0.133437,0.773355,0.698201,0.444265,-0.001915,0.392821,0.698133,0.484334,0.140691
1,ADL,ADL_258_clip_06,1,0.016667,0.999278,0.749849,0.813340,-0.138927,0.999355,0.750998,...,0.545988,0.150652,0.331871,0.733151,0.562751,0.168333,0.325424,0.734614,0.615252,0.196572
2,ADL,ADL_258_clip_06,2,0.033333,0.999675,0.775521,0.784802,-0.157145,0.999730,0.775811,...,0.558782,0.159259,0.556027,0.690598,0.457576,0.245180,0.328392,0.691842,0.510915,0.332946
3,ADL,ADL_258_clip_06,3,0.050000,0.999524,0.744846,0.811618,-0.166858,0.999563,0.745004,...,0.571575,0.167866,0.308033,0.733747,0.541699,0.201393,0.376794,0.728171,0.589217,0.166611
4,ADL,ADL_258_clip_06,4,0.066667,0.999344,0.745148,0.821267,-0.216221,0.999298,0.746944,...,0.617362,0.127735,0.263840,0.761638,0.618134,0.134159,0.292943,0.742217,0.646395,0.127203
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
334611,FALL,S_D_0236_clip_00,25,0.833333,0.999983,0.434626,0.471936,-0.311287,0.999980,0.438283,...,0.742590,-0.189421,0.659698,0.486615,0.691948,0.085916,0.981766,0.332682,0.721123,-0.245745
334612,FALL,S_D_0236_clip_00,26,0.866667,0.999983,0.442541,0.488367,-0.343633,0.999970,0.447242,...,0.721588,-0.133398,0.557207,0.457308,0.712772,0.158664,0.975555,0.330984,0.702318,-0.203823
334613,FALL,S_D_0236_clip_00,27,0.900000,0.999380,0.454446,0.499925,-0.202630,0.999454,0.458607,...,0.714285,-0.078113,0.558282,0.398406,0.694079,-0.057877,0.865539,0.320859,0.681110,-0.128284
334614,FALL,S_D_0236_clip_00,28,0.933333,0.999671,0.469536,0.517396,-0.200646,0.999712,0.473724,...,0.715408,-0.116903,0.798776,0.392183,0.691846,-0.144201,0.891496,0.314040,0.674383,-0.159454


In [ ]:
nulls = df_processed.isnull().sum().to_frame(name='null_count')
display(nulls)

,null_count
video,0
file_id,0
frame,0
timestamp,0
kp0_visibility,0
...,...
kp31_z,9668
kp32_visibility,0
kp32_x,0
kp32_y,285


In [ ]:
# 만약 처리 후 NaN 값이 존재한다면 아래 코드 실행
# pandas로 후처리

# video, field_id 별로 비선형 보간 처리 - Cubic Spline(3차 스플라인) 보간법
numeric_cols = df_processed.select_dtypes(include='number').columns

df_filled = (
    df_processed
    .groupby([df['video'], df['file_id'].str[:2]], group_keys=False)
    .apply(lambda g: g.assign(**{
        col: g[col].interpolate(method='cubic', limit_direction='both').ffill().bfill()
        for col in numeric_cols
    }))
    .reset_index(drop=True)
)


In [ ]:
nulls = df_filled.isnull().sum().to_frame(name='null_count')
display(nulls)

,null_count
video,0
file_id,0
frame,0
timestamp,0
kp0_visibility,0
...,...
kp31_z,0
kp32_visibility,0
kp32_x,0
kp32_y,0


In [ ]:
nan_info = df_filled.isna().sum()
nan_cols = nan_info[nan_info > 0]

nan_cols

,0


In [ ]:
df_processed.to_csv("/content/drive/MyDrive/OnSafe/no_nan_df.csv", index=False, encoding="utf-8")